# 08. Évaluation Finale, Efficacité Oracle et Rétro-Test

Projet : PharmaBénin — Plateforme Intelligente de Gestion Pharmaceutique  
Date : 1er octobre 2026 | Responsable : R2 (Modélisation)

> **Règle absolue** : toutes les métriques affichées ci-dessous proviennent de
> `ml/reports/metrics.json` généré par `ml/evaluate.py`.
> **Aucune valeur n'est écrite en dur.**
> Régénérer avec : `python ml/evaluate.py --split test --out ml/reports/metrics.json`


In [1]:
import json
import os

# Charger les métriques calculées par ml/evaluate.py
metrics_path = os.path.join('..', 'reports', 'metrics.json')
with open(metrics_path, 'r', encoding='utf-8') as f:
    metrics = json.load(f)

g = metrics['global']

header = f"""
====================================================
 MÉTRIQUES FINALES — {metrics['model_version']} ({metrics['split']})
 Données simulées seed=42  |  {metrics['generated_at'][:19]}Z
 SHA-256 données : {metrics['data_hash'][:16]}…
====================================================
"""
print(header)

def fmt(v, pct=False):
    if v is None: return 'N/A'
    return f"{v*100:.1f} %" if pct else f"{v:.4f}"

rows = [
    ('WAPE (test)',              fmt(g['wape'])),
    ('WAPE naïf saisonnier',     fmt(g['wape_naive'])),
    ('MASE',                     fmt(g['mase'])),
    ('RMSSE',                    fmt(g['rmsse'])),
    ('Biais relatif',            fmt(g['bias'])),
    ('Couverture q10–q90',        fmt(g['coverage_q10_q90'], pct=True)),
    ('Perte Pinball q10',         fmt(g['pinball_q10'])),
    ('Perte Pinball q90',         fmt(g['pinball_q90'])),
    ('WAPE Oracle',              fmt(g['oracle_wape'])),
    ('Efficacité (WAPE/Oracle)', fmt(g['efficiency'])),
    ('IC 95% WAPE',              str(g['wape_ci_95'])),
]

print(f'{"Métrique":<30} {"Valeur":>15}')
print('-' * 47)
for name, val in rows:
    print(f'{name:<30} {val:>15}')



 MÉTRIQUES FINALES — demand_lightgbm_v1.0.0 (test)
 Données simulées seed=42  |  2026-10-01T22:35:19Z
 SHA-256 données : 09706088c053ebea…

Métrique                             Valeur
-----------------------------------------------
WAPE (test)                          0.9443
WAPE naïf saisonnier                 0.8646
MASE                                 1.0922
RMSSE                                1.0176
Biais relatif                        -0.1269
Couverture q10–q90                    59.3 %
Perte Pinball q10                    8.2478
Perte Pinball q90                   33.7811
WAPE Oracle                          0.6287
Efficacité (WAPE/Oracle)             1.5020
IC 95% WAPE                          [0.9149, 0.9721]


## 2. Analyse honnête des résultats

### Ce qui fonctionne
- **Antipaludiques** : MASE = 0.92 — le modèle bat le naïf sur cette famille. Le signal CHIRPS (pluie retardée de 4 sem.) est bien exploité.
- **Alibori, Atacora, Borgou, Plateau, Zou** : MASE < 1.
- **Calibration CQR** : couverture = 80.0 % sur le jeu de calibration (target atteint).

### Ce qui ne fonctionne pas encore
- **MASE global = 1.09** : sur le jeu de test, le modèle ne bat pas le naïf saisonnier en médiane.
- **Couverture test = 59.3 %** : au lieu de 80 %. La calibration CQR ne se transfère pas bien.
- **Collines : WAPE = 255.7 %** : anomalie — épidémie simulée dans la période test.
- **Biais = -12.7 %** : sous-estimation systématique.

### Causes probables
1. **Données simulées** — 140 semaines, structure trop régulière pour un bon apprentissage des chocs.
2. **Hyperparamètres légers** — n_estimators=200, num_leaves=31 pour CPU rapide.
3. **Test décalé** — les chocs (épidémies, ruptures) dans juil–sept 2026 ne sont pas bien capturés.

### Pistes d'amélioration
- Augmenter l'historique simulé à 3–5 ans.
- Recalibrer CQR séparément par type d'établissement.
- Régler les hyperparamètres avec Optuna.


In [2]:
# Performances par type d'établissement
by_type = metrics.get('by_facility_type', {})
print('\n--- Performance par type d\'établissement ---')
print(f'{"Type":<20} {"WAPE":>8} {"MASE":>8} {"Couverture":>12}')
print('-' * 52)
for ftype, m in sorted(by_type.items()):
    w = m['wape'] if m['wape'] else 0
    mase = m['mase'] if m['mase'] else 0
    cov = m['coverage_q10_q90'] if m['coverage_q10_q90'] else 0
    print(f'{ftype:<20} {w:>8.4f} {mase:>8.4f} {cov*100:>11.1f}%')

# Performances par famille de produits
by_fam = metrics.get('by_product_family', {})
print('\n--- Performance par famille thérapeutique ---')
print(f'{"Famille":<20} {"WAPE":>8} {"MASE":>8} {"Couverture":>12}')
print('-' * 52)
for fam, m in sorted(by_fam.items(), key=lambda x: x[1]['wape'] if x[1]['wape'] else 999):
    w = m['wape'] if m['wape'] else 0
    mase = m['mase'] if m['mase'] else 0
    cov = m['coverage_q10_q90'] if m['coverage_q10_q90'] else 0
    print(f'{fam:<20} {w:>8.4f} {mase:>8.4f} {cov*100:>11.1f}%')



--- Performance par type d'établissement ---
Type                    WAPE     MASE   Couverture
----------------------------------------------------
health_center         1.0253   1.1227       62.9%
hospital              0.8867   1.0683       51.1%

--- Performance par famille thérapeutique ---
Famille                 WAPE     MASE   Couverture
----------------------------------------------------
antipaludiques        0.7643   0.9189       56.8%
vaccins               0.8143   1.0146       60.4%
autres                1.0294   1.1356       61.1%
antibiotiques         1.0384   1.2172       58.3%
maternels             1.4131   1.5596       58.3%
